In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

# Read raw products
df_products = spark.read.parquet(f"{BRONZE}/products")

# Check schema and total records
df_products.printSchema()
print("Total products:", df_products.count())

# Find inconsistent category names
display(
    df_products.groupBy("category")
    .count()
    .orderBy("category")
)


root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

Total products: 60


category,count
Home & Kitchen,2
BOOKS,1
Beauty,9
Books,8
Electronics,8
Electronics,1
FASHION,2
Fashion,8
Home & Kitchen,6
Home & kitchen,1


In [0]:

from pyspark.sql import functions as F

# Step 1: Define the six standard categories
CATEGORY_MAP = {
    "electronics": "Electronics",
    "home and kitchen": "Home & Kitchen",
    "fashion": "Fashion",
    "books": "Books",
    "beauty": "Beauty",
    "sports": "Sports"
}

# Step 2: Convert Python dictionary to Spark Map
category_map = F.create_map(
    *[F.lit(x) for pair in CATEGORY_MAP.items() for x in pair]
)

# Step 3: Normalize raw category values
category_key = F.lower(
    F.regexp_replace(
        F.trim(
            F.regexp_replace(F.col("category"), "&", " and ")
        ),
        r"\s+", " "
    )
)

# Step 4: Replace raw categories with standard values
products_clean = df_products.withColumn(
    "category",
    category_map[category_key]
)

# Step 5: Verify results
display(
    products_clean
    .groupBy("category")
    .count()
    .orderBy("category")
)


category,count
Beauty,10
Books,10
Electronics,10
Fashion,10
Home & Kitchen,10
Sports,10


In [0]:

# Inspect product names before cleaning

name_profile = (
    df_products
    .select("product_id", "product_name")
    .withColumn(
        "spaces_cleaned",
        F.regexp_replace(
            F.trim(F.col("product_name")),
            r"\s+", " "
        )
    )
    .withColumn(
        "title_case_candidate",
        F.initcap(F.col("spaces_cleaned"))
    )
)

display(name_profile.orderBy("product_id"))


product_id,product_name,spaces_cleaned,title_case_candidate
1,Wireless Earbuds,Wireless Earbuds,Wireless Earbuds
2,Bluetooth Speaker,Bluetooth Speaker,Bluetooth Speaker
3,Power Bank 20000mAh,Power Bank 20000mAh,Power Bank 20000mah
4,USB-C Charger 65W,USB-C Charger 65W,Usb-c Charger 65w
5,Smartwatch,Smartwatch,Smartwatch
6,Wireless Mouse,Wireless Mouse,Wireless Mouse
7,Mechanical Keyboard,Mechanical Keyboard,Mechanical Keyboard
8,Webcam 1080p,Webcam 1080p,Webcam 1080p
9,Noise Cancelling Headphones,Noise Cancelling Headphones,Noise Cancelling Headphones
10,Portable SSD 1TB,Portable SSD 1TB,Portable Ssd 1tb


In [0]:

# Clean product names without changing their casing
products_clean = products_clean.withColumn(
    "product_name",
    F.regexp_replace(
        F.trim(F.col("product_name")),
        r"\s+",
        " "
    )
)

# Verify cleaned product names
display(
    products_clean.select(
        "product_id", "product_name", "category"
    ).orderBy("product_id")
)


product_id,product_name,category
1,Wireless Earbuds,Electronics
2,Bluetooth Speaker,Electronics
3,Power Bank 20000mAh,Electronics
4,USB-C Charger 65W,Electronics
5,Smartwatch,Electronics
6,Wireless Mouse,Electronics
7,Mechanical Keyboard,Electronics
8,Webcam 1080p,Electronics
9,Noise Cancelling Headphones,Electronics
10,Portable SSD 1TB,Electronics


In [0]:

from pyspark.sql import functions as F

# Check column data types
products_clean.printSchema()

# Validate cleaned product records
result = products_clean.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("product_id").alias("unique_ids"),
    F.countDistinct("category").alias("categories"),
    F.sum(F.col("product_id").isNull().cast("int")).alias("null_ids"),
    F.sum(F.col("category").isNull().cast("int")).alias("null_categories"),
    F.sum(
        (
            F.col("product_name").isNull() |
            (F.trim(F.col("product_name")) == "")
        ).cast("int")
    ).alias("invalid_names"),
    F.sum(
        (
            F.col("unit_price").isNull() |
            (F.col("unit_price") <= 0)
        ).cast("int")
    ).alias("invalid_prices")
).first()

print(result.asDict())

# Stop if data quality checks fail
assert result["total_rows"] == 60
assert result["unique_ids"] == 60
assert result["categories"] == 6
assert result["null_ids"] == 0
assert result["null_categories"] == 0
assert result["invalid_names"] == 0
assert result["invalid_prices"] == 0

print("Products validation passed!")


root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

{'total_rows': 60, 'unique_ids': 60, 'categories': 6, 'null_ids': 0, 'null_categories': 0, 'invalid_names': 0, 'invalid_prices': 0}
Products validation passed!


In [0]:

products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("atliq_dl.silver.products")

print("Products Silver table created!")

# Read back the persisted table
display(
    spark.table("atliq_dl.silver.products")
    .groupBy("category")
    .count()
    .orderBy("category")
)


Products Silver table created!


category,count
Beauty,10
Books,10
Electronics,10
Fashion,10
Home & Kitchen,10
Sports,10


In [0]:
%run ./utils_dq

In [0]:
raw = spark.read.parquet(f"{BRONZE}/products")
silver = spark.table("atliq_dl.silver.products")

r = raw.dropDuplicates(["product_id"])
j = r.alias("r").join(silver.alias("s"), "product_id")

log_dq("products", "category_standardized",
       j.filter(~F.col("r.category").eqNullSafe(F.col("s.category"))).count())

log_dq("products", "product_name_cleaned",
       j.filter(~F.col("r.product_name").eqNullSafe(F.col("s.product_name"))).count())

log_dq("products", "duplicate_product_ids_removed",
       raw.count() - r.count())

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'products'")
    .orderBy("rule")
)

run_date,table_name,rule,rows_affected
2026-11-01,products,category_standardized,13
2026-10-31,products,category_standardized,13
2026-10-31,products,duplicate_product_ids_removed,0
2026-11-01,products,duplicate_product_ids_removed,0
2026-10-31,products,product_name_cleaned,5
2026-11-01,products,product_name_cleaned,5
